# GhostChef AI — Day 2: Data Merging & Master Dataset Construction

Build a single master dataset by merging sales, calendar, and price data.
Transform wide-format sales into long format, then enrich with temporal and pricing features.

# Load Raw Data

**Objective:** Ingest all source CSV files into memory.

Load the five raw datasets: sales (validation & evaluation), calendar events, sell prices, and sample submission.

In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
sell_prices = pd.read_csv('../../data/raw/sell_prices.csv')
sample_submission = pd.read_csv('../../data/raw/sample_submission.csv')
sales_train_validation = pd.read_csv('../../data/raw/sales_train_validation.csv')
sales_train_evaluation = pd.read_csv('../../data/raw/sales_train_evaluation.csv')
calendar = pd.read_csv('../../data/raw/calendar.csv')

# Data Understanding

**Objective:** Inspect structure and schema of each dataset.

Preview the first rows of each source table to confirm columns, types, and granularity before merging.

In [5]:
sell_prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [6]:
sample_submission.head()

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
sales_train_validation.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [8]:
sales_train_evaluation.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [9]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


# Wide to Long Transformation

**Objective:** Reshape the wide-format sales matrix into a tidy long-format table.

Each row in `sales_train_validation` has 1,913 day columns (d_1 … d_1913).
`pd.melt` unpivots these into one row per (item, day) pair, producing ~58M rows.

In [10]:
ID_COLUMNS = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

In [11]:
DAY_COLUMNS = sorted(
    (col for col in sales_train_validation.columns if col.startswith("d_")),
    key=lambda col: int(col.split("_", 1)[1])
)

In [12]:
sales_long = pd.melt(
    sales_train_validation,
    id_vars=ID_COLUMNS,
    value_vars=DAY_COLUMNS,
    var_name="day",
    value_name="sales"
)

In [13]:
sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [14]:
sales_long.shape

(58327370, 8)

# Merge with Calendar

**Objective:** Enrich each sales row with calendar features.

Join on `day` ↔ `d` to attach date, weekday, events, SNAP flags, and `wm_yr_wk`.
This adds 14 columns; the result is a 22-column dataframe at 58M rows.

In [15]:
sales_calendar = sales_long.merge(
    calendar,
    left_on="day",
    right_on="d",
    how="left"
)

In [16]:
sales_calendar.shape

(58327370, 22)

In [17]:
sales_calendar.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0


In [18]:
sales_calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 22 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   id            object
 1   item_id       object
 2   dept_id       object
 3   cat_id        object
 4   store_id      object
 5   state_id      object
 6   day           object
 7   sales         int64 
 8   date          object
 9   wm_yr_wk      int64 
 10  weekday       object
 11  wday          int64 
 12  month         int64 
 13  year          int64 
 14  d             object
 15  event_name_1  object
 16  event_type_1  object
 17  event_name_2  object
 18  event_type_2  object
 19  snap_CA       int64 
 20  snap_TX       int64 
 21  snap_WI       int64 
dtypes: int64(8), object(14)
memory usage: 9.6+ GB


In [19]:
sales_calendar.isnull().sum()

id                     0
item_id                0
dept_id                0
cat_id                 0
store_id               0
state_id               0
day                    0
sales                  0
date                   0
wm_yr_wk               0
weekday                0
wday                   0
month                  0
year                   0
d                      0
event_name_1    53631910
event_type_1    53631910
event_name_2    58205410
event_type_2    58205410
snap_CA                0
snap_TX                0
snap_WI                0
dtype: int64

# Merge with Sell Prices

**Objective:** Attach the selling price for each (item, store, week) combination.

Join on `item_id` + `store_id` + `wm_yr_wk` to add `sell_price`.
Nulls appear when an item was not available for sale in a given week (~21% of rows).

In [20]:
master_df = sales_calendar.merge(
    sell_prices,
    on=["item_id", "store_id", "wm_yr_wk"],
    how="left"
)

In [21]:
print(master_df.shape)

(58327370, 23)


In [22]:
master_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN


In [23]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 23 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            object 
 1   item_id       object 
 2   dept_id       object 
 3   cat_id        object 
 4   store_id      object 
 5   state_id      object 
 6   day           object 
 7   sales         int64  
 8   date          object 
 9   wm_yr_wk      int64  
 10  weekday       object 
 11  wday          int64  
 12  month         int64  
 13  year          int64  
 14  d             object 
 15  event_name_1  object 
 16  event_type_1  object 
 17  event_name_2  object 
 18  event_type_2  object 
 19  snap_CA       int64  
 20  snap_TX       int64  
 21  snap_WI       int64  
 22  sell_price    float64
dtypes: float64(1), int64(8), object(14)
memory usage: 10.0+ GB


In [24]:
master_df.isnull().sum().sort_values(ascending=False)

event_type_2    58205410
event_name_2    58205410
event_type_1    53631910
event_name_1    53631910
sell_price      12299413
month                  0
snap_WI                0
snap_TX                0
snap_CA                0
d                      0
year                   0
id                     0
item_id                0
weekday                0
wm_yr_wk               0
date                   0
sales                  0
day                    0
state_id               0
store_id               0
cat_id                 0
dept_id                0
wday                   0
dtype: int64

# Validation

**Objective:** Verify data integrity of the final master dataset.

Check for duplicate rows and review numeric column statistics.
Event columns have expected nulls (most days are not events); `sell_price` nulls indicate items not yet available for sale.

In [25]:
master_df.duplicated().sum()

0

In [26]:
master_df.describe()

,sales,wm_yr_wk,wday,month,year,snap_CA,snap_TX,snap_WI,sell_price
count,5.832737e+07,5.832737e+07,5.832737e+07,5.832737e+07,5.832737e+07,5.832737e+07,5.832737e+07,5.832737e+07,4.602796e+07
mean,1.126322e+00,1.133919e+04,3.997386e+00,6.357554e+00,2.013209e+03,3.293257e-01,3.293257e-01,3.293257e-01,4.408089e+00
std,3.873108e+00,1.503742e+02,2.000652e+00,3.458773e+00,1.532086e+00,4.699684e-01,4.699684e-01,4.699684e-01,3.403657e+00
min,0.000000e+00,1.110100e+04,1.000000e+00,1.000000e+00,2.011000e+03,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e-02
25%,0.000000e+00,1.121700e+04,2.000000e+00,3.000000e+00,2.012000e+03,0.000000e+00,0.000000e+00,0.000000e+00,2.180000e+00
50%,0.000000e+00,1.133300e+04,4.000000e+00,6.000000e+00,2.013000e+03,0.000000e+00,0.000000e+00,0.000000e+00,3.470000e+00
75%,1.000000e+00,1.144800e+04,6.000000e+00,9.000000e+00,2.015000e+03,1.000000e+00,1.000000e+00,1.000000e+00,5.840000e+00
max,7.630000e+02,1.161300e+04,7.000000e+00,1.200000e+01,2.016000e+03,1.000000e+00,1.000000e+00,1.000000e+00,1.073200e+02


# Save Final Master Dataset

**Objective:** Persist the merged dataset to disk as a single CSV.

This is the only save operation in the notebook. All intermediate results are kept in memory.

In [29]:
master_df["sales"].head()

0    0
1    0
2    0
3    0
4    0
Name: sales, dtype: int64

In [27]:
import math
import os

os.makedirs("../../data/interim/day2_draft_chunks", exist_ok=True)

chunk_size = 5_000_000   # 5 مليون صف في كل ملف
n_chunks = math.ceil(len(master_df) / chunk_size)

print(f"Number of files: {n_chunks}")

Number of files: 12


<>:4: SyntaxWarning: invalid escape sequence '\g'
<>:4: SyntaxWarning: invalid escape sequence '\g'
C:\Users\mohme\AppData\Local\Temp\ipykernel_21864\483042234.py:4: SyntaxWarning: invalid escape sequence '\g'
  os.makedirs("D:\gearofproml\day 2", exist_ok=True)


In [28]:
chunk_size = 5000

for start in range(0, len(sales_train_validation), chunk_size):

    chunk = sales_train_validation.iloc[start:start+chunk_size]

    melted = pd.melt(
        chunk,
        id_vars=ID_COLUMNS,
        value_vars=DAY_COLUMNS,
        var_name="day",
        value_name="sales"
    )

    # Merge
    # Preprocessing
    # Save

    del melted